# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_AUTH_TOKEN')
# DeepSeek's OpenAI-compatible endpoint is the same host without the /anthropic suffix
base_url = os.getenv('ANTHROPIC_BASE_URL', 'https://api.deepseek.com').rstrip('/').removesuffix('/anthropic')
MODEL = 'deepseek-chat'

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-"):
    print("An API key was found, but it doesn't start sk-; please check you're using the right key - see troubleshooting notebook")
else:
    print(f"API key found and looks good so far! Using {base_url} with model {MODEL}")

# Point the OpenAI client at the .env endpoint explicitly, so it can't fall back to api.openai.com
openai = OpenAI(api_key=api_key, base_url=base_url)

API key found and looks good so far! Using https://api.deepseek.com with model deepseek-chat


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'avatar page', 'url': 'https://edwarddonner.com/avatar/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling deepseek-chat
Found 7 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-chat
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling deepseek-chat
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
speridlabs/idol
Updated
11 days ago
•
24.7k
•
1.47k
xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
Updated
1 day ago
•
11.8k
•
955
moonshotai/Kimi-K3.1-Pro
Updated
4 days ago
•
27.9k
•
1.76k
ricursiveAI/Ripple-65B-Preview
Updated
3 days

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-chat
Found 10 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nSearch models, datasets, users...\nCtrl K\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nsperidlabs/idol\nUpdated\n11 days ago\n•\n24.7k\n•\n1.47k\nxtools-at/DeepSeek-

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-chat
Found 11 relevant links


# Hugging Face — Company Brochure

## The AI Community Building the Future

Hugging Face is the platform where the machine learning community collaborates on models, datasets, and applications. We are the home of machine learning — a place to create, discover, and collaborate on ML, better.

---

## What We Do

- **Models** — Browse and host 2M+ machine learning models, from community experiments to state-of-the-art releases.
- **Datasets** — Explore 500k+ datasets spanning everything from security research to pixel-art atlases and aggregated news.
- **Spaces** — Discover 1M+ ai applications. Build, demo, and share ML apps directly on the platform.
- **Buckets** — Durable storage for repositories, builds, and agent memory.
- **Tools & Libraries** — We maintain core open-source tooling behind modern AI, including the `transformers` library and the `hf` CLI / `huggingface_hub` ecosystem.
- **Research & Thought Leadership** — Our blog and Daily Papers track the pulse of open-source AI, from agentic RL to the state of open source.

---

## Why Hugging Face

- **The collaboration platform** — Host and collaborate on unlimited public models, datasets, and applications.
- **Move faster** — Purpose-built infrastructure so teams can go from idea to shipped ML faster.
- **Open by default** — We champion open-source AI and the global ecosystem that powers it.

---

## For Teams & Enterprise

- **Team & Enterprise** plans for organizations collaborating at scale
- **Enterprise Support** for mission-critical deployments
- **Inference Providers & Inference Endpoints** for production-grade model serving
- **Hugging Face PRO** for individual power users

---

## Company Culture

Hugging Face is a **community-first, open-source-minded** company. Our identity is built around collaboration — a "Team" organization whose mission is literally *the AI community building the future*. We publish openly (research articles, papers, and tooling), maintain foundational open-source libraries, and celebrate the builders in our ecosystem through featured Spaces, models, and community activity feeds. Our brand reflects this energy: warm, friendly, and unmistakably community-driven.

---

## Our Community

- **109,000+ followers** on the platform
- Researchers, hobbyists, enterprises, and open-source contributors worldwide
- Active ecosystem of trending models, Spaces, datasets, Discord, and forum discussions
- Featured work spans coding agents, video generation, 3D reconstruction, Mixture-of-Experts training, and more

---

## Careers

We're growing. Visit our careers page for current openings across engineering, research, community, and more — and join a team that's building the future of AI in the open.

---

## Brand & Contact

- **Website:** huggingface.co
- **Brand assets:** Logos (SVG, PNG, AI), brand colors (#FFD21E, #FF9D00, #6B7280), and official bio available on our Brand Assets page
- **Find us:** Discord, Forum, GitHub, Blog, and Daily Papers

**Hugging Face — The AI community building the future.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-chat
Found 11 relevant links


# Hugging Face

**The AI community building the future.**

Hugging Face is the collaboration platform for the machine learning community — the place where researchers, developers, and organizations come together to create, discover, and share AI models, datasets, and applications.

---

## What We Offer

**The Home of Machine Learning**
Create, discover, and collaborate on machine learning better.

**The Collaboration Platform**
Host and collaborate on unlimited public models, datasets, and applications.

**Move Faster**
Access a massive, growing ecosystem so you can build on the work of others instead of starting from scratch.

**Explore and Build**
- **2M+ models** to browse and use
- **1M+ applications** (Spaces)
- **500k+ datasets** for training and research

**Enterprise Solutions**
- Team & Enterprise plans
- Enterprise Support
- Inference Providers and Inference Endpoints
- Storage Buckets
- Hugging Face PRO

---

## By the Numbers

- 2M+ models available
- 1M+ applications hosted
- 500k+ datasets
- A community of over 100,000 followers on the platform alone (109,912 and counting)

---

## Community & Culture

Hugging Face is, at its core, **a community**. Our mission statement — *"The AI community building the future"* — reflects a culture built on openness, collaboration, and shared progress.

- **Open by default:** We champion open-source AI, from models and datasets to tooling and research.
- **Collaborative:** Unlimited public hosting for models, datasets, and apps means everyone can contribute and build on each other's work.
- **Knowledge-sharing:** Our blog, papers, Daily Papers, forum, and Discord keep the community learning together — covering everything from agent training and tokenization to the state of open-source AI.
- **Global perspective:** We actively document and celebrate the worldwide open-source AI ecosystem, from DeepSeek to AI+.

---

## Who We Serve

- **Machine learning researchers** exploring new architectures and methods
- **Developers** building AI applications and agents
- **Enterprises** deploying models at scale with support, endpoints, and storage
- **The broader open-source AI community** — organizations, hobbyists, and learners worldwide

---

## Careers & Getting Involved

While specific job listings aren't detailed here, Hugging Face is a growing, community-driven company. Prospective recruits and contributors can engage through:

- **The platform itself** — create an account and start publishing models, datasets, and Spaces
- **Community channels** — Discord, the Forum, and GitHub
- **Learning resources** — our Learn section and blog articles
- **Enterprise contact** — for organizations interested in Team & Enterprise solutions

*Join the AI community building the future.*

---

## Brand

Official logos and colors are available on our Brand Assets page. Brand colors include **#FFD21E**, **#FF9D00**, and **#6B7280**.

**Website:** https://huggingface.co

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-chat
Found 13 relevant links


# Hugging Face — The AI Community Building the Future

## Who We Are

Hugging Face is **the collaboration platform for the machine learning community** — the place where developers, researchers, and organizations come together to create, discover, and share AI. From models and datasets to interactive applications, we provide the open infrastructure that powers modern machine learning.

Our community spans the globe, and our platform is the home of open-source AI: a hub where the world's leading AI builders publish their work and where newcomers learn by exploring real, runnable examples.

## What We Offer

**A Platform for Every Step of the ML Journey**

- **Models** — Browse and share from over **2 million models**, from cutting-edge language models to specialized vision and audio systems.
- **Datasets** — Access **500,000+ datasets** for training, evaluation, and research.
- **Spaces & Applications** — Explore and deploy **1 million+ interactive AI applications**, from demos to full products.
- **Buckets, Inference Endpoints & Providers** — Flexible infrastructure to store, serve, and scale your AI workloads.

**For Enterprises & Teams**

Hugging Face offers Team & Enterprise solutions, Enterprise Support, inference endpoints, and storage buckets — giving organizations the tools to move faster while collaborating on unlimited public models, datasets, and applications.

## Who We Serve

- **AI researchers and developers** building and sharing state-of-the-art models
- **Enterprises** deploying AI at scale with dedicated support and infrastructure
- **Startups and individual builders** prototyping and launching AI applications
- **The open-source community** — anyone contributing to the collective future of AI

Our platform is home to work from organizations large and small, and our community spans hundreds of thousands of active members collaborating in the open.

## Our Culture

Hugging Face is rooted in **openness, collaboration, and community**. We believe the future of AI is best built in the open — where models, data, and ideas are shared freely and built upon by everyone.

- **Community-first** — Our tagline says it all: *the AI community building the future.*
- **Open by default** — We champion open-source AI and transparent collaboration.
- **Always learning** — Our blog, papers, courses, and guides keep the community at the frontier, from hands-on Mixture-of-Experts training guides to coding agents and evaluation research.
- **Playful and approachable** — From our signature 🤗 emoji to our friendly, human-centered brand, we make cutting-edge AI accessible to all.

## Learn & Grow With Us

Hugging Face supports learners and practitioners at every level through:

- **Learn** resources, courses, and tutorials
- **Blog** articles on the latest research and engineering
- **Daily Papers** and community papers
- **Discord and Forum** communities for discussion and support

## Careers & Community

While our careers page isn't included here, Hugging Face is a mission-driven, community-centered organization where you'd be working on the frontier of open AI. Get involved today:

- **Join the community** — Sign up and start exploring models, datasets, and Spaces
- **Connect** — Follow us on [Discord](https://discord.com/invite/huggingface), our [Forum](https://discuss.huggingface.co), and [GitHub](https://github.com/huggingface)
- **Stay informed** — Read our [blog](https://huggingface.co/blog) and track daily papers
- **For enterprises** — Explore Team & Enterprise solutions and dedicated support

---

**Hugging Face** — *The AI community building the future.*

🌐 [huggingface.co](https://huggingface.co)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>